# Convlutional Neural Networks

In this tutorial, you will build a convolutional neural network using the data pipeline you created over the Oxford 102 flower datase in the previous notebook.

## Importing Packages

In [26]:
import os
from scipy.io import loadmat         # For loading .mat (MATLAB) files
from PIL import Image                # For image loading and processing

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.v2 as transforms_v2
import torch.nn as nn

## Data Preparation

In [15]:
dataset_folder = "./datasets/oxford_102_flowers"

**Image Transformations**

In [ ]:
# Composing image transformations comprising the following sequence of steps

transforms = transforms_v2.Compose([
    # Step 1: Resizes height and width of each image to 224 pixels
    transforms_v2.Resize(256),
    transforms_v2.CenterCrop(224),

    # Step 2: Converts each image to a tensor
    transforms_v2.ToImage(),

    # Step 3: Converts pixel value type from integer to float and normalizes pixel values
    transforms_v2.ToDtype(torch.float32, scale=True),
])

In [23]:
class Oxford102FlowersDataset(Dataset):
    def __init__(self, root_dir, transforms=None):
        self.root_dir = root_dir
        self.transforms = transforms

        self.image_dir = os.path.join(root_dir, "jpg")
        self.labels_file = os.path.join(root_dir, "imagelabels.mat")

        # Loads images file names (loading images will be delayed until __getitem__ is called)
        self.image_files = [f for f in os.listdir(self.image_dir) if f.endswith(".jpg")]

        # Loads labels
        mat_content = loadmat(self.labels_file)
        self.labels = mat_content["labels"][0] - 1  # Adjusts labels to start from 0

    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.image_files)
    
    def __getitem__(self, idx):
        """Retrieves the image and label at the specified index."""
        image_file_name = f"image_{idx + 1:05d}.jpg"
        image_file_path = os.path.join(self.image_dir, image_file_name)
        image = Image.open(image_file_path)
        label = self.labels[idx]

        if (self.transforms):
            image = self.transforms(image)

        return image, label

In [24]:
# Creates the Oxford 102 Flowers dataset
dataset = Oxford102FlowersDataset(
    root_dir=dataset_folder,
    transforms=transforms,
)

**Splitting Data**

Splitting the the full dataset into training, validation and test set.

The model gets trained on training set, validation set helps checking performance during training and to tune model paraneters while the test set is for final check on the trained model performance.

In [27]:
train_set, val_set, test_set = random_split(dataset, [0.7, 0.15, 0.15])

**Batching Data**

In [28]:
train_set_loader = DataLoader(
    dataset=train_set,
    batch_size=32, 
    shuffle=True)

val_set_loader = DataLoader(
    dataset=val_set,
    batch_size=32, 
    shuffle=False)

test_set_loader = DataLoader(
    dataset=test_set,
    batch_size=32, 
    shuffle=False)

## Modeling

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # First convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=3,      # Number of input channels (e.g., 1 for grayscale images and 3 for RGB images)
            out_channels=32,    # Number of output channels (filters)
            kernel_size=3,      # Each filter is 3x3 (meaning 9 weights per filter) that slides over the image and responds to different patterns
            padding=1
        )
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)    # Reduces spatial dimention of the input by half

        # Second convolutional layer
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Third convolutional layer
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Flatten layer
        self.flatten = nn.Flatten() # Flattens the 2D feature maps into a 1D vector to feed into the fully connected layer

        # Fully connected layer
        self.fc1 = nn.Linear(in_features=128 * 4 * 4,           # Last layer has 64 channels of size 4x4
                            out_features=512                    # Output size of the fully connected layer
                            )
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)    # Dropout layer for regularization (dropout rate typically ranges from 0.2 to 0.5)

        self.fc2 = nn.Linear(in_features=512, out_features=15)  # Final output layer for 15 classes
        
    def forward(self, x):
        # First convolutional layer
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Second convolutional layer
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Flatten layer
        x = self.flatten(x)

        # Fully connected layer
        x = self.fc(x)

        return x

In [5]:
# Creates an instance of the SimpleCNN model
model = SimpleCNN()
print(model)

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc): Linear(in_features=3136, out_features=10, bias=True)
)


## Model Training

[TODO] 
- Showing changes in shape of input data when flows through the CNN layers.
- Training over 10 epochs
- Plotting learning curves [losses, accuracy]

## Model Evaluation

## Conclusion